# SIEE Shadow RGB upper body, YOLO11n v2

Choose **Runtime → Change runtime type → T4 GPU**. Run cells in order.
The 230 training images contain two reviewed sessions. The 120 validation
images are a different capture session, but were already inspected and
used for diagnosis. Validation is for development and model selection,
**not** an independent estimate of generalization. Gather a new blind
session with projector off before claiming performance or using motion.

Training starts from the preserved v1 `best.pt` and uses `resume=False`.
No cameras, motors, UART or ROS commands are run here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import tarfile

FOLDER = Path('/content/drive/MyDrive/SIE/colab_runs/ar0234_shadow_yolo11n_v2')
PARTS = {
    's1': ('shadow_rgb_colab_v2_train_s1_20261031.tar.gz', '9a6a65a800c0847bdac646cd05c631de92aeacc976226160f9ec720ea03e4ebb', 238),
    's2': ('shadow_rgb_colab_v2_train_s2_20261031.tar.gz', '76550684e0826f210d93989a62f233443ae12af179066365e6bd53f38c191feb', 224),
    'val': ('shadow_rgb_colab_v2_val_20261031.tar.gz', '80b74757e35c0a76a7d13f21c43bc0f9ef3813c5af0f4156053c9a26ed706c6c', 240),
}
OLD_BEST = Path('/content/drive/MyDrive/SIE/colab_runs/ar0234_person_upper_body_yolo11n_v1/train_run/weights/best.pt')
DATA_ROOT = Path('/content/sie_shadow_v2')
DATA_YAML = DATA_ROOT / 'dataset.yaml'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

assert OLD_BEST.is_file(), f'Missing v1 checkpoint: {OLD_BEST}'
for name, (filename, expected, _) in PARTS.items():
    path = FOLDER / filename
    assert path.is_file(), f'Missing dataset part: {path}'
    assert sha256(path) == expected, f'DATA_ARCHIVE_SHA256_MISMATCH:{name}'
print('Three archives SHA-256 verified; old best.pt:', sha256(OLD_BEST))

In [ ]:
# Keep the known training stack version; Colab supplies the CUDA-enabled PyTorch.
%pip install -q 'ultralytics==8.4.152' 'onnx>=1.17,<2'
import torch, ultralytics
print('torch', torch.__version__, 'ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Refuse unexpected archive members, verify all 350 original JPEG bytes.
assert not DATA_ROOT.exists(), 'Remove or rename previous /content/sie_shadow_v2'
DATA_ROOT.mkdir()
extracted_names = set()
for name, (filename, _, expected_count) in PARTS.items():
    with tarfile.open(FOLDER / filename, 'r:gz') as tf:
        members = tf.getmembers()
        assert len(members) == expected_count, (name, len(members))
        assert all(m.isfile() and not m.name.startswith('/') and
                   '..' not in Path(m.name).parts and
                   m.name not in extracted_names for m in members)
        extracted_names.update(m.name for m in members)
        tf.extractall(DATA_ROOT, filter='data')
assert len(extracted_names) == 702

manifest = json.loads((DATA_ROOT / 'bundle_manifest.json').read_text())
assert manifest['sha256_overlap'] == 0
assert manifest['train']['images'] == 230 and manifest['val']['images'] == 120
for name, expected in manifest['per_image_sha256'].items():
    assert sha256(DATA_ROOT / name) == expected, name
for split, count in (('train', 230), ('val', 120)):
    images = sorted((DATA_ROOT / 'images' / split).glob('*.jpg'))
    labels = sorted((DATA_ROOT / 'labels' / split).glob('*.txt'))
    assert len(images) == len(labels) == count
    assert {p.stem for p in images} == {p.stem for p in labels}
print('Verified train 230, development val 120; disjoint source hashes')
print(DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO
import shutil

RUN_ROOT = Path('/content/runs')
RUN_NAME = 'ar0234_shadow_person_upper_body_yolo11n_v2'
model = YOLO(str(OLD_BEST))
result = model.train(
    data=str(DATA_YAML), epochs=100, patience=20,
    imgsz=640, batch=16, seed=20260915,
    device=0, workers=2, optimizer='auto',
    project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False,
    save_period=10, plots=True, resume=False,
)
RUN = Path(model.trainer.save_dir)
NEW_BEST = RUN / 'weights' / 'best.pt'
assert NEW_BEST.is_file()
print('New checkpoint:', NEW_BEST, sha256(NEW_BEST))

In [ ]:
# Both checkpoints see the same 120 validation images under the same settings.
# These numbers are development diagnostics, since this split guides selection.
old_model, new_model = YOLO(str(OLD_BEST)), YOLO(str(NEW_BEST))
ap = {}
for label, net in (('v1', old_model), ('v2', new_model)):
    m = net.val(data=str(DATA_YAML), split='val', imgsz=640,
                device=0, plots=False, project=str(RUN_ROOT),
                name=RUN_NAME + '_' + label + '_dev_val')
    ap[label] = {'precision': float(m.box.mp), 'recall': float(m.box.mr),
                 'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map)}
print(json.dumps(ap, indent=2))

In [ ]:
# Frame-level presence and localization at the existing 0.40 operating threshold.
# This re-runs v1 via PyTorch, so it can differ from the earlier ONNX pipeline.
VAL_IMAGES = sorted((DATA_ROOT / 'images/val').glob('*.jpg'))
def overlap(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x1-x0) * max(0, y1-y0)
    aa = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    bb = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (aa + bb - inter) if aa + bb > inter else 0.0

def frame_metrics(net):
    total = dict(positive=0, negative=0, tp_presence=0, tp_iou50=0, fp_images=0)
    for image in VAL_IMAGES:
        label = DATA_ROOT / 'labels/val' / (image.stem + '.txt')
        gt = label.read_text().strip()
        p = net.predict(source=str(image), imgsz=640, conf=0.40,
                        iou=0.45, device=0, verbose=False)[0]
        boxes = p.boxes.xyxy.cpu().numpy().tolist()
        if not gt:
            total['negative'] += 1
            total['fp_images'] += bool(boxes)
            continue
        total['positive'] += 1
        class_id, cx, cy, w, h = map(float, gt.split())
        assert class_id == 0
        iw, ih = p.orig_shape[1], p.orig_shape[0]
        actual = ((cx-w/2)*iw, (cy-h/2)*ih,
                  (cx+w/2)*iw, (cy+h/2)*ih)
        total['tp_presence'] += bool(boxes)
        total['tp_iou50'] += any(overlap(box, actual) >= 0.50 for box in boxes)
    return total

frames = {'v1': frame_metrics(old_model), 'v2': frame_metrics(new_model)}
print(json.dumps(frames, indent=2))

In [ ]:
# Export with the v1 ONNX input contract; still verify downstream preprocessing.
import onnx
exported = Path(YOLO(str(NEW_BEST)).export(
    format='onnx', imgsz=640, dynamic=False, simplify=False,
    opset=17, nms=False, device=0,
))
onnx.checker.check_model(onnx.load(str(exported)))
DEST = FOLDER / RUN_NAME
assert not DEST.exists(), f'Already exists: {DEST}'
DEST.mkdir()
shutil.copy2(NEW_BEST, DEST / 'best.pt')
shutil.copy2(exported, DEST / 'best.onnx')
shutil.copy2(RUN / 'args.yaml', DEST / 'args.yaml')
summary = {
    'scope': 'SHADOW_RGB_DEVELOPMENT_ONLY',
    'dataset_parts_sha256': {name: digest for name, (_, digest, _) in PARTS.items()},
    'v1_checkpoint_sha256': sha256(OLD_BEST),
    'v2_checkpoint_sha256': sha256(NEW_BEST),
    'v2_onnx_sha256': sha256(exported),
    'ultralytics_version': ultralytics.__version__,
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'ap': ap, 'frames_conf_0p40_iou_0p50': frames,
    'limits': 'Validation was previously inspected; new blind projector-off session needed before any motion use.',
}
(DEST / 'comparison.json').write_text(json.dumps(summary, indent=2) + chr(10))
print('Saved:', DEST)
print(json.dumps(summary, indent=2))